In [1]:
import polars as pl
from pathlib import Path
from unidecode import unidecode
BASE_DIR = Path.cwd().parent / 'data_input' / 'nao_consignados'

In [93]:
retorno = pl.read_excel(f'{BASE_DIR}/RETORNO TODOS 06.2026.xlsx')
base_ativos = pl.read_excel(f'{BASE_DIR}/BASE ATIVOS E CANCELADOS.xlsx')
coopart = pl.read_excel(f'{BASE_DIR}/ANALITICO COPARTICIPAÇÃO SIGRH - 06.2026 (consignado) COMPLETO.xlsx')
remuneracao = pl.read_excel(f'{BASE_DIR}/REMUNERAÇÃO 05.2026.xlsx')


Could not determine dtype for column 22, falling back to string


In [94]:
def formata_coluna(cols:list) -> list:
    cols_norm = []
    for col in cols:
        cols_norm.append(unidecode(col.lower().replace(' ', '_')).replace('-', ''))

    return cols_norm

retorno.columns = formata_coluna(retorno.columns)
base_ativos.columns = formata_coluna(base_ativos.columns)
remuneracao.columns = formata_coluna(remuneracao.columns)
coopart.columns = formata_coluna(coopart.columns)

retorno =  retorno.select(pl.all().exclude(['mesclado', 'qtde_rubricas']))


In [95]:
retorno

orgao,ano,mes,matricula,cpf__titular,nome_titular,rubrica,prazo,valor,rubrica_principal
str,i64,i64,str,str,str,str,str,f64,str
"""001""",2026,6,"""00311405""","""015210995372""","""JOAO JOSE DE OLIVEIRA""","""41134""","""999""",578.0,"""41134"""
"""001""",2026,6,"""00405302""","""056321945153""","""IVETE ALVES DA ROCHA""","""41134""","""999""",578.0,"""41134"""
"""001""",2026,6,"""00416428""","""052455173100""","""MIRIAM GOMES DE MELO""","""41134""","""999""",605.8,"""41134"""
"""001""",2026,6,"""0043034X""","""053924207100""","""ELIAS JOSE DE SENA JUNIOR""","""41134""","""999""",723.19,"""41134"""
"""001""",2026,6,"""00435740""","""048394246168""","""PATRICIA VIANA FERREIRA""","""41134""","""999""",578.0,"""41134"""
…,…,…,…,…,…,…,…,…,…
"""992""",2026,6,"""02486830""","""000342824163""","""BENILSON SOUZA NUNES""","""41136""","""001""",471.56,"""41136"""
"""992""",2026,6,"""0257182X""","""002399919157""","""LUIS FERNANDO PORTELA DE FARIA""","""41134""","""999""",578.0,"""41134"""
"""992""",2026,6,"""0257182X""","""002399919157""","""LUIS FERNANDO PORTELA DE FARIA""","""41136""","""001""",501.97,"""41136"""


### Retorno folha

In [96]:
retorno = retorno.with_columns([
    pl.col('cpf__titular').str.slice(-11)
]).with_columns([
    pl.concat_str([pl.col('orgao'), pl.col('cpf__titular'), pl.col('matricula'), pl.col('rubrica')], separator='_').alias('chave')
])

### Coopart 

In [97]:
coopart = coopart.with_columns([
    pl.col('cpf').str.replace_all(r'[^\d]', '')
]).rename({'copart_mes_titular':'titular', 'copart_mes_dependente':'dependente'})

In [98]:
rubrica = coopart.unpivot(
    index=['emp_cod', 'matricula', 'cpf'],
    on=['dependente', 'titular'],
    variable_name='tipo_beneficiario',
    value_name='valor'
).with_columns([
    pl.when(pl.col('tipo_beneficiario') == 'dependente')
    .then(pl.lit('41144')).otherwise(pl.lit('41136')).alias('rubrica')
]).with_columns([
    pl.concat_str([pl.col('emp_cod'), pl.col('cpf'), pl.col('matricula'), pl.col('rubrica')], separator='_').alias('chave')
])

### Base ativos

In [99]:
base_ativos = base_ativos.with_columns([
    pl.col('cpf_titular').str.replace_all(r'[^\d]', ''),
    pl.col('cpf_segurado').str.replace_all(r'[^\d]', '')
]).select(['cpf_titular', 'cpf_segurado', 'tipo_segurado', 'data_adesao', 'codigo_da_empresa', 'matricula_titular_8_digitos', 'idade']).rename({
    'codigo_da_empresa':'orgao', 'matricula_titular_8_digitos':'matricula'
})

base_ativos = base_ativos.with_columns([
    pl.when(pl.col('tipo_segurado') == 'TITULAR').then(pl.lit('41134'))
    .when(pl.col('idade') <= 18).then(pl.lit('41135'))
    .when(pl.col('idade') <= 23).then(pl.lit('41171'))
    .when(pl.col('idade') <= 28).then(pl.lit('41175'))
    .when(pl.col('idade') <= 33).then(pl.lit('41191'))
    .when(pl.col('idade') <= 38).then(pl.lit('41192'))
    .when(pl.col('idade') <= 43).then(pl.lit('41193'))
    .when(pl.col('idade') <= 48).then(pl.lit('41121'))
    .when(pl.col('idade') <= 53).then(pl.lit('41122'))
    .when(pl.col('idade') <= 58).then(pl.lit('41123'))
    .otherwise(pl.lit('41224')).alias('rubrica')
]).with_columns([
    pl.when(pl.col('rubrica') == '41134').then(0)
    .when(pl.col('rubrica') == '41135').then(225)
    .when(pl.col('rubrica') == '41171').then(250)
    .when(pl.col('rubrica') == '41175').then(340)
    .when(pl.col('rubrica') == '41191').then(370)
    .when(pl.col('rubrica') == '41192').then(420)
    .when(pl.col('rubrica') == '41193').then(479)
    .when(pl.col('rubrica') == '41121').then(530)
    .when(pl.col('rubrica') == '41122').then(657)
    .when(pl.col('rubrica') == '41123').then(764)
    .otherwise(853).alias('mensalidade')
])

In [104]:
retorno

orgao,ano,mes,matricula,cpf__titular,nome_titular,rubrica,prazo,valor,rubrica_principal,chave
str,i64,i64,str,str,str,str,str,f64,str,str
"""001""",2026,6,"""00311405""","""15210995372""","""JOAO JOSE DE OLIVEIRA""","""41134""","""999""",578.0,"""41134""","""001_15210995372_00311405_41134"""
"""001""",2026,6,"""00405302""","""56321945153""","""IVETE ALVES DA ROCHA""","""41134""","""999""",578.0,"""41134""","""001_56321945153_00405302_41134"""
"""001""",2026,6,"""00416428""","""52455173100""","""MIRIAM GOMES DE MELO""","""41134""","""999""",605.8,"""41134""","""001_52455173100_00416428_41134"""
"""001""",2026,6,"""0043034X""","""53924207100""","""ELIAS JOSE DE SENA JUNIOR""","""41134""","""999""",723.19,"""41134""","""001_53924207100_0043034X_41134"""
"""001""",2026,6,"""00435740""","""48394246168""","""PATRICIA VIANA FERREIRA""","""41134""","""999""",578.0,"""41134""","""001_48394246168_00435740_41134"""
…,…,…,…,…,…,…,…,…,…,…
"""992""",2026,6,"""02486830""","""00342824163""","""BENILSON SOUZA NUNES""","""41136""","""001""",471.56,"""41136""","""992_00342824163_02486830_41136"""
"""992""",2026,6,"""0257182X""","""02399919157""","""LUIS FERNANDO PORTELA DE FARIA""","""41134""","""999""",578.0,"""41134""","""992_02399919157_0257182X_41134"""
"""992""",2026,6,"""0257182X""","""02399919157""","""LUIS FERNANDO PORTELA DE FARIA""","""41136""","""001""",501.97,"""41136""","""992_02399919157_0257182X_41136"""


In [108]:
base_ativos.filter(pl.col('cpf_titular') == '02399919157')

cpf_titular,cpf_segurado,tipo_segurado,data_adesao,orgao,matricula,idade,rubrica,mensalidade
str,str,str,date,str,str,i64,str,i32
"""02399919157""","""02399919157""","""TITULAR""",2024-11-06,"""992""","""0257182X""",37,"""41134""",0
